In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import sys
import os



from sklearn.model_selection import train_test_split


import json

import math

sys.path.append(os.path.abspath("../../dossier_donnees/jour2/scripts"))

import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Reshape, GRU, Conv1D, MaxPooling1D, Bidirectional, Dropout, GlobalMaxPooling1D, Flatten, AveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.metrics import RootMeanSquaredError

import argparse
import random

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


2024-11-15 00:14:14.486384: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-15 00:14:20.447288: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-11-15 00:14:20.447351: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2024-11-15 00:14:21.235254: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-15 00:14:33.104752: W tensorflow/stream_executor/platform/de

In [3]:
def get_json_stat(min_max_path):
    """
    Cette fonction permet de lire le fichier json défini dans min_max_path et retourne les 4 valeurs qui y sont écrites
    :param min_max_path: Chemin vers le fichier json de description des données
    :type min_max_path: string
    :return: 4 valeurs (min, max, moyenne et std)
    """
    with open(min_max_path) as json_file:
        data = json.load(json_file)
    return data['min'], data['max'], data['moyenne'], data['std']

In [4]:
def eval_model(pred_csv, y_csv, min_max_path):
    """
    Cette fonction charge les prédictions et les valeurs réelles à partir de fichiers CSV, 
    puis utilise les statistiques de normalisation (min, max) provenant d'un fichier JSON pour normaliser les erreurs. 
    Elle calcule ensuite la moyenne de l'erreur quadratique normalisée entre les prédictions et les valeurs réelles.
    param pred_csv (str): Chemin vers le fichier CSV contenant les prédictions du modèle.
    param y_csv (str): Chemin vers le fichier CSV contenant les valeurs réelles.
    param min_max_path (str): Chemin vers le fichier JSON contenant les valeurs minimales et maximales utilisées pour la normalisation des données.
    returns: None: Affiche simplement la valeur de nRMSE calculée.
    """
    pred_test = pd.read_csv(pred_csv, sep=',', header=None, index_col=False).to_numpy()
    y_test = pd.read_csv(y_csv, sep=',', header=None, index_col=False).to_numpy() 
    min_d, max_d, moyenne, std = get_json_stat(min_max_path)
    min_val, max_val = int(min_d), int(max_d)
    global_nrmse = 0.0
    for pred, gt in zip(pred_test, y_test):
        global_nrmse += (gt, pred) / (max_val - min_val)
    global_nrmse /= len(pred_test)
    print(f"global test nRMSE: {global_nrmse:.8f}")

In [5]:
def create_sequences_with_targets(data, input_steps=60, output_steps=5):
    X, y = [], []
    for i in range(len(data) - input_steps - output_steps):
        X.append(data[i:i + input_steps])  # Fenêtre de 60 éléments
        y.append(data[i + input_steps:i + input_steps + output_steps])  # Cibles : 5 éléments suivants
    return np.array(X), np.array(y)

In [6]:
def nrmse_loss(y_true, y_pred):

    
    moyenne = 66120.59
    std = 68177.18
    min = 176
    max = 381085
    y_true = (y_true * std) + moyenne
    y_pred = (y_pred * std) + moyenne
    # Calcul de l'erreur quadratique moyenne (MSE)
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    
    # Calcul de la racine carrée de la MSE pour obtenir la RMSE
    rmse = tf.sqrt(mse)
    
    # Normalisation en divisant par la différence max-min des vraies valeurs
    rmse /= (max - min)
    return rmse

In [ ]:
preproced = '../../dossier_donnees/jour2/data/preprocessed_data'
raw = '../../dossier_donnees/jour2/data/raw_data'
description_json = '../../dossier_donnees/jour2/data/description_data.json'

In [ ]:
# Normalisation 
with open(description_json, 'r') as fic:
    json_desc = json.load(fic)

train_data = pd.read_csv(raw + '/raw_train.csv')
valid_data = pd.read_csv(raw + '/raw_valid.csv')



In [ ]:
x_train = pd.read_csv(preproced + '/x_train.csv', header=None)
y_train = pd.read_csv(preproced + '/y_train.csv', header=None)
x_valid = pd.read_csv(preproced + '/x_valid.csv', header=None)
y_valid = pd.read_csv(preproced + '/y_valid.csv', header=None)
x_test = pd.read_csv(preproced + '/x_test.csv', header=None)

x_train = x_train.apply(lambda x: (x - json_desc['min']) / (json_desc['max'] - json_desc['min']))
y_train = y_train.apply(lambda x: (x - json_desc['min']) / (json_desc['max'] - json_desc['min']))
x_valid = x_valid.apply(lambda x: (x - json_desc['min']) / (json_desc['max'] - json_desc['min']))
y_valid = y_valid.apply(lambda x: (x - json_desc['moyenne']) / json_desc['std'])
x_test = x_test.apply(lambda x: (x - json_desc['moyenne']) / json_desc['std'])


In [77]:
new_x_train = pd.read_csv('new_x_train.csv', header=None)
new_y_train = pd.read_csv('new_y_train.csv', header=None, index_col=None).drop(0, axis=1)
x_valid = pd.read_csv('x_valid.csv', header=None)
y_valid = pd.read_csv('y_valid.csv', header=None)

mini = min(new_x_train.min())
maxi = max(new_x_train.max())
mean = new_x_train.mean()
std = new_x_train.std()

new_x_train = (new_x_train - mini) / (maxi - mini)
new_y_train = (new_y_train - mini) / (maxi - mini)
x_valid = (x_valid - mini) / (maxi - mini)



In [78]:
print(mini)

176


In [79]:
new_y_train.head()

,1
0,0.120218
1,0.005125
2,0.048558
3,0.003014
4,0.201539


In [301]:
X, y = create_sequences_with_targets(train_data['Bits/s'].values)
X_df = pd.DataFrame(X)
y_df = pd.DataFrame(y)

X_df = X_df.apply(lambda x: (x - json_desc['min']) / (json_desc['max'] - json_desc['min']))
y_df = y_df.apply(lambda x: (x - json_desc['min']) / (json_desc['max'] - json_desc['min']))

In [ ]:
print(np.shape(x_train))
print(np.shape(y_train))

print(np.shape(X))
print(np.shape(y))

In [357]:
X_df = pd.read_csv(raw + '/raw_train.csv')
X_df = X_df['Bits/s']
X_df = X_df.loc[:100000]

V_df = pd.read_csv(raw + '/raw_valid.csv')
V_df = V_df['Bits/s']
V_df = V_df.loc[:5000]

In [358]:
mean = X_df.mean()
std = X_df.std()

mini = X_df.min()
maxi = X_df.max()

X_df = (X_df - mini) / (maxi - mini)
V_df = (V_df - mini) / (maxi - mini)

X_df = X_df.to_numpy()
V_df = V_df.to_numpy()

In [359]:
X = [X_df[t:t+60] for t in range(len(X_df)-61)]
y = [X_df[t+60:t+61] for t in range(len(X_df)-61)]
X = pd.DataFrame(X)
y = pd.DataFrame(y)

V = [V_df[t:t+60] for t in range(len(V_df)-61)]
u = [V_df[t+60:t+65] for t in range(len(V_df)-61)]
V = pd.DataFrame(V)
u = pd.DataFrame(u)

In [338]:
X.head()

,0,1,2,3,4,5,6,7,8,9,...,50,51,52,53,54,55,56,57,58,59
0,0.599849,0.030054,0.053577,0.014366,0.186186,0.020362,0.216262,0.063889,0.340323,0.161550,...,0.021790,0.085039,0.071093,0.100370,0.081321,0.000200,0.061096,0.003591,0.008138,0.108708
1,0.030054,0.053577,0.014366,0.186186,0.020362,0.216262,0.063889,0.340323,0.161550,0.214266,...,0.085039,0.071093,0.100370,0.081321,0.000200,0.061096,0.003591,0.008138,0.108708,0.120218
2,0.053577,0.014366,0.186186,0.020362,0.216262,0.063889,0.340323,0.161550,0.214266,0.043506,...,0.071093,0.100370,0.081321,0.000200,0.061096,0.003591,0.008138,0.108708,0.120218,0.005125
3,0.014366,0.186186,0.020362,0.216262,0.063889,0.340323,0.161550,0.214266,0.043506,0.902767,...,0.100370,0.081321,0.000200,0.061096,0.003591,0.008138,0.108708,0.120218,0.005125,0.048558
4,0.186186,0.020362,0.216262,0.063889,0.340323,0.161550,0.214266,0.043506,0.902767,1.000000,...,0.081321,0.000200,0.061096,0.003591,0.008138,0.108708,0.120218,0.005125,0.048558,0.003014


In [ ]:
model_tf = Sequential([
    Input(shape=(60, 1)),
    LSTM(64),
    Dense(1)
])


model_tf.compile(optimizer='adam', loss='mse')
model_tf.summary()


Model: "sequential_19"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_11 (LSTM)              (None, 64)                16896     
                                                                 
 dense_26 (Dense)            (None, 8)                 520       
                                                                 
 dense_27 (Dense)            (None, 1)                 9         
                                                                 
 dense_28 (Dense)            (None, 1)                 2         
                                                                 
 dense_29 (Dense)            (None, 1)                 2         
                                                                 
Total params: 17,429
Trainable params: 17,429
Non-trainable params: 0
_________________________________________________________________


In [94]:
new_x_train.head()

,0,1,2,3,4,5,6,7,8,9,...,50,51,52,53,54,55,56,57,58,59
0,0.599849,0.030054,0.053577,0.014366,0.186186,0.020362,0.216262,0.063889,0.340323,0.161550,...,0.021790,0.085039,0.071093,0.100370,0.081321,0.000200,0.061096,0.003591,0.008138,0.108708
1,0.030054,0.053577,0.014366,0.186186,0.020362,0.216262,0.063889,0.340323,0.161550,0.214266,...,0.085039,0.071093,0.100370,0.081321,0.000200,0.061096,0.003591,0.008138,0.108708,0.120218
2,0.053577,0.014366,0.186186,0.020362,0.216262,0.063889,0.340323,0.161550,0.214266,0.043506,...,0.071093,0.100370,0.081321,0.000200,0.061096,0.003591,0.008138,0.108708,0.120218,0.005125
3,0.014366,0.186186,0.020362,0.216262,0.063889,0.340323,0.161550,0.214266,0.043506,0.902767,...,0.100370,0.081321,0.000200,0.061096,0.003591,0.008138,0.108708,0.120218,0.005125,0.048558
4,0.186186,0.020362,0.216262,0.063889,0.340323,0.161550,0.214266,0.043506,0.902767,1.000000,...,0.081321,0.000200,0.061096,0.003591,0.008138,0.108708,0.120218,0.005125,0.048558,0.003014


In [97]:
new_y_train[0:1000].head()

,1
0,0.120218
1,0.005125
2,0.048558
3,0.003014
4,0.201539


In [90]:
with tf.device('/GPU:0'):
    model_tf.fit(new_x_train[0:1000], new_y_train[0:1000], epochs=1, verbose=1)

model_tf.save('tf_model.keras')

32/32 [==============================] - 4s 77ms/step - loss: 0.0126


In [91]:
print(resultat)

[]


In [92]:
resultat = []
for _, row in x_valid.iterrows():

    r = []
    for i in range(5):
        res = model_tf.predict(row, verbose=0)
        print(res)
        break

        temp = row.to_numpy()[1:]
        temp = np.append(temp, res)
        row = pd.DataFrame(temp)
        
        r.append(res)
    break
    resultat.append(r)

print(np.shape(res))

[[0.04673452]
 [0.03823813]
 [0.03957914]
 [0.05267318]
 [0.05525143]
 [0.04115848]
 [0.04079189]
 [0.04765851]
 [0.04337431]
 [0.0387493 ]
 [0.04458807]
 [0.04515766]
 [0.04075403]
 [0.04195438]
 [0.04294172]
 [0.04153538]
 [0.04163921]
 [0.04790342]
 [0.04920661]
 [0.04679607]
 [0.04488419]
 [0.04127587]
 [0.04507793]
 [0.04157889]
 [0.03818085]
 [0.04528855]
 [0.0413868 ]
 [0.04408322]
 [0.04158469]
 [0.04081227]
 [0.03821451]
 [0.03870485]
 [0.03893527]
 [0.04478268]
 [0.0392817 ]
 [0.05295046]
 [0.04991757]
 [0.04721396]
 [0.04017816]
 [0.04545745]
 [0.05177253]
 [0.04405345]
 [0.03843109]
 [0.0414878 ]
 [0.03842637]
 [0.04453324]
 [0.04214995]
 [0.04198101]
 [0.03818085]
 [0.03847089]
 [0.03821865]
 [0.04953541]
 [0.03988986]
 [0.04667352]
 [0.04270316]
 [0.0509282 ]
 [0.03834525]
 [0.04909337]
 [0.04274178]
 [0.03843315]]
(60, 1)


In [ ]:
print(resultat[0:5])

In [328]:
res_ajus = [[int((x * (maxi - mini) + mini)) for x in t] for t in res]
df = pd.DataFrame(res_ajus)

valid_ajus = [[int((x * (maxi - mini) + mini)) for x in t] for t in u.to_numpy()]
df2 = pd.DataFrame(valid_ajus)

In [329]:

# Sauvegarde en CSV
df.to_csv('resultat.csv', index=False, header=None)

df2.to_csv('valid_test.csv', index=False, header=None)

In [330]:
eval_model('resultat.csv', 'valid_test.csv', description_json)

global test nRMSE: 54.63878598


In [121]:
df.head()

,0,1,2,3,4
0,1133680,1237745,1204977,1205800,1234284
1,1070037,1173026,1138358,1142139,1171976
2,1072729,1176466,1141179,1145054,1174780
3,1095330,1198695,1165111,1166951,1196846
4,1140036,1244400,1211741,1211621,1240455


In [ ]:
with tf.device('/GPU:0'):
    model_tf.fit(x_train, y_train, epochs=20, verbose=1)

In [ ]:
# Couches de convolution
x = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(inputs)
x = Conv1D(filters=128, kernel_size=3, activation='relu', padding='same')(x)
x = Dropout(0.2)(x)

# Couches LSTM empilées
x = LSTM(128, return_sequences=True)(x)
x = Dropout(0.2)(x)
x = LSTM(64)(x)

# Couche Dense pour la prédiction
outputs = Dense(5)(x)



Conv1D(64, kernel_size=3, activation='relu', padding='same'),
Conv1D(128, kernel_size=3, activation='relu', padding='same'),
Dropout(0.2),

LSTM(128, return_sequences=True),
Dropout(0.2)
LSTM(64),

Dense(5)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, LSTM, Dense, Dropout
from tensorflow.keras.models import Model
from keras_tuner import RandomSearch

def build_model(hp):
    inputs = Input(shape=(60, 1))
    
    # Choix du nombre de filtres et de couches Conv1D
    x = inputs
    for i in range(hp.Int('conv_layers', min_value=1, max_value=3)):
        filters = hp.Choice(f'filters_{i}', values=[32, 64, 128])
        kernel_size = hp.Choice(f'kernel_size_{i}', values=[3, 5])
        x = Conv1D(filters=filters, kernel_size=kernel_size, activation='relu', padding='same')(x)
        if hp.Boolean(f'dropout_conv_{i}'):
            x = Dropout(0.2)(x)

    # Choix du nombre de couches et unités LSTM
    for i in range(hp.Int('lstm_layers', min_value=1, max_value=2)):
        units = hp.Choice(f'lstm_units_{i}', values=[64, 128, 256])
        x = LSTM(units, return_sequences=(i < hp.Int('lstm_layers', min_value=1, max_value=2) - 1))(x)
        if hp.Boolean(f'dropout_lstm_{i}'):
            x = Dropout(0.2)(x)

    # Couche de sortie Dense
    outputs = Dense(5)(x)
    model = Model(inputs=inputs, outputs=outputs)
    
    # Compilation du modèle
    model.compile(optimizer='adam', loss='mse')
    
    return model

# Configurer le tuner
tuner = RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=20,  # Nombre d'essais différents
    executions_per_trial=2,  # Nombre de fois pour chaque combinaison
    directory='my_dir',
    project_name='time_series_prediction'
)

tuner.search(x_train, y_train, epochs=2)

# Récupérer le meilleur modèle
best_model = tuner.get_best_models(num_models=1)[0]